# Pi+ Decay Chain Analysis
**π+ → μ+ → e+ (Michel) in water — TOF, range, and Cherenkov study**

Data from Geant4 simulation using FTFP_BERT physics with Frank-Tamm Cherenkov.
Pion momenta sampled from Delta resonance decay kinematics (GENIE-style sample).

## ROOT file structure
- **EventTree**: one row per event — summary of pi+, mu+, Michel e+ track
- **StepTree**: one row per step — full kinematics at every tracking step

In [ ]:
import uproot
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pandas as pd
from scipy import stats

plt.rcParams.update({'figure.dpi': 120, 'font.size': 12})

# ── Load ROOT file ────────────────────────────────────────────────────────────
ROOTFILE = 'pion_decay.root'   # copy from server: scp dfleming@Sol:~/piongun/piondecay/build/pion_decay.root .

f = uproot.open(ROOTFILE)
print('Trees:', f.keys())

# Load EventTree
ev = f['EventTree'].arrays(library='pd')
print(f'EventTree: {len(ev):,} events')
print(ev.columns.tolist())

# Load StepTree (can be large -- filter to pi+ only first if needed)
st = f['StepTree'].arrays(library='pd')
print(f'StepTree: {len(st):,} steps')
print(st['particle'].value_counts())

## 1. Pion momentum distribution (input check)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
M_PI = 0.13957
N_WATER = 1.34
P_CHER_THR = M_PI / np.sqrt(N_WATER**2 - 1)

ax.hist(ev['pi_p0'], bins=80, range=(0, 1.0),
        color='steelblue', alpha=0.8, label='Input pi+ momenta')
ax.axvline(P_CHER_THR, color='red', ls='--', lw=1.5,
           label=f'Cherenkov threshold {P_CHER_THR:.3f} GeV/c')
ax.set_xlabel('|p_pi+| (GeV/c)')
ax.set_ylabel('Events')
ax.set_title('Input pi+ momentum distribution (Delta decay sample)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('01_pion_momentum.png', dpi=150)
plt.show()

## 2. Track length: pi+, mu+, Michel e+

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

particles = [
    ('pi_track_len', 'pi+',       'steelblue',  (0, 3000)),
    ('mu_track_len', 'mu+',       'darkorange',  (0, 10000)),
    ('e_track_len',  'Michel e+', 'forestgreen', (0, 3000)),
]

for ax, (col, label, color, xlim) in zip(axes, particles):
    data = ev[col][ev[col] > 0]
    ax.hist(data / 10, bins=80, range=(xlim[0]/10, xlim[1]/10),
            color=color, alpha=0.8)
    ax.set_xlabel('Track length (cm)')
    ax.set_ylabel('Events')
    ax.set_title(f'{label} track length in water')
    ax.grid(alpha=0.3)
    print(f'{label}: mean={data.mean()/10:.1f} cm, '
          f'median={data.median()/10:.1f} cm, '
          f'max={data.max()/10:.1f} cm')

plt.suptitle('Track lengths in water (20m sphere)', fontsize=13)
plt.tight_layout()
plt.savefig('02_track_lengths.png', dpi=150)
plt.show()

## 3. Lab-frame decay times

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# pi+ decay time
t_pi = ev['pi_time_decay'][ev['pi_time_decay'] > 0]
axes[0].hist(t_pi, bins=100, color='steelblue', alpha=0.8)
axes[0].set_xlabel('Lab-frame decay time (ns)')
axes[0].set_ylabel('Events')
axes[0].set_title('pi+ decay time (lab frame)')
axes[0].grid(alpha=0.3)
print(f'pi+ decay time: mean={t_pi.mean():.3f} ns, '
      f'median={t_pi.median():.3f} ns')

# mu+ decay time (includes pion flight time)
t_mu = ev['mu_time_decay'][ev['mu_time_decay'] > 0]
axes[1].hist(t_mu, bins=100, color='darkorange', alpha=0.8)
axes[1].set_xlabel('Lab-frame time at mu+ decay (ns)')
axes[1].set_ylabel('Events')
axes[1].set_title('mu+ decay time (lab frame, includes pi+ flight)')
axes[1].grid(alpha=0.3)
print(f'mu+ decay time: mean={t_mu.mean():.2f} ns, '
      f'median={t_mu.median():.2f} ns')

plt.tight_layout()
plt.savefig('03_decay_times.png', dpi=150)
plt.show()

## 4. Cherenkov photon yields per particle

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

cher_data = [
    ('pi_cher_total', 'pi+',       'steelblue'),
    ('mu_cher_total', 'mu+',       'darkorange'),
    ('e_cher_total',  'Michel e+', 'forestgreen'),
]

for ax, (col, label, color) in zip(axes, cher_data):
    data = ev[col][ev[col] > 0]
    ax.hist(data, bins=80, color=color, alpha=0.8)
    ax.set_xlabel('Cherenkov photons (Frank-Tamm)')
    ax.set_ylabel('Events')
    ax.set_title(f'{label} Cherenkov yield')
    ax.grid(alpha=0.3)
    print(f'{label}: mean={data.mean():.0f}, '
          f'median={data.median():.0f}, '
          f'max={data.max():.0f} photons')

plt.suptitle('Cherenkov photon yields (Frank-Tamm, 300-700 nm)', fontsize=13)
plt.tight_layout()
plt.savefig('04_cherenkov_yields.png', dpi=150)
plt.show()

# Combined
print(f'\nTotal Cherenkov (pi+mu+e): '
      f'mean={ev["total_cher"].mean():.0f}, '
      f'median={ev["total_cher"].median():.0f}')

## 5. Cherenkov yield vs pion momentum

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

pairs = [
    ('pi_cher_total', 'pi+ Cherenkov vs p_pi0', 'steelblue'),
    ('mu_cher_total', 'mu+ Cherenkov vs p_pi0', 'darkorange'),
    ('total_cher',    'Total Cherenkov vs p_pi0','purple'),
]

for ax, (ycol, title, color) in zip(axes, pairs):
    mask = ev[ycol] > 0
    ax.hexbin(ev['pi_p0'][mask], ev[ycol][mask],
              gridsize=50, cmap='Blues', mincnt=1)
    ax.set_xlabel('Initial pi+ momentum (GeV/c)')
    ax.set_ylabel('Cherenkov photons')
    ax.set_title(title)
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('05_cher_vs_momentum.png', dpi=150)
plt.show()

## 6. Step-level: KE vs track length (Bragg-like curves)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (pname, color, title) in zip(axes, [
        ('pi+',  'steelblue',  'pi+ KE vs track length'),
        ('mu+',  'darkorange', 'mu+ KE vs track length'),
        ('e+',   'forestgreen','Michel e+ KE vs track length'),
    ]):
    data = st[st['particle'] == pname]
    if len(data) == 0:
        ax.set_title(f'{title} (no data)')
        continue

    ax.hexbin(data['track_len'] / 10, data['KE'] * 1000,
              gridsize=60, cmap='hot_r', mincnt=1)
    ax.set_xlabel('Cumulative track length (cm)')
    ax.set_ylabel('Kinetic energy (MeV)')
    ax.set_title(title)
    ax.grid(alpha=0.2)

plt.suptitle('Energy vs track length (Bethe-Bloch / Bragg behavior)', fontsize=13)
plt.tight_layout()
plt.savefig('06_KE_vs_tracklength.png', dpi=150)
plt.show()

## 7. Step-level: Cherenkov rate (dN/dx) vs beta

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Frank-Tamm prediction for comparison
beta_arr = np.linspace(0.75, 1.0, 200)
N_water = 1.34
alpha = 1/137.036
# Integrated over 300-700 nm: ~490 photons/cm for beta=1, z=1
def frank_tamm_dndx(beta, z=1, n=1.34, lam_min=300e-7, lam_max=700e-7):
    """photons per cm"""
    if beta * n <= 1: return 0
    sin2 = 1 - 1/(beta**2 * n**2)
    return 2 * np.pi * alpha * z**2 * sin2 * (1/lam_min - 1/lam_max)

ft_pred = np.array([frank_tamm_dndx(b) for b in beta_arr])

for ax, (pname, color) in zip(axes, [
        ('pi+', 'steelblue'),
        ('mu+', 'darkorange'),
        ('e+',  'forestgreen'),
    ]):
    data = st[(st['particle'] == pname) & (st['step_len'] > 0.1)]
    if len(data) == 0:
        continue
    dndx = data['cher_step'] / (data['step_len'] / 10)  # per cm
    mask = (dndx > 0) & (data['beta'] > 0.7)
    ax.hexbin(data['beta'][mask], dndx[mask],
              gridsize=50, cmap='Blues', mincnt=1)
    ax.plot(beta_arr, ft_pred, 'r--', lw=1.5, label='Frank-Tamm (z=1)')
    ax.set_xlabel('beta (v/c)')
    ax.set_ylabel('dN_cher/dx (photons/cm)')
    ax.set_title(f'{pname} Cherenkov rate vs beta')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('07_cher_rate_vs_beta.png', dpi=150)
plt.show()

## 8. Decay vertex positions (pi+ and mu+)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# pi+ decay z position vs initial momentum
mask_pi = ev['pi_z_decay'].abs() < 1e6  # remove outliers
axes[0].hexbin(ev['pi_p0'][mask_pi],
               ev['pi_z_decay'][mask_pi] / 1000,  # mm -> m
               gridsize=50, cmap='Blues', mincnt=1)
axes[0].set_xlabel('Initial pi+ momentum (GeV/c)')
axes[0].set_ylabel('pi+ decay z position (m)')
axes[0].set_title('pi+ decay vertex vs momentum')
axes[0].grid(alpha=0.2)

# mu+ decay z position
mask_mu = ev['mu_z_decay'].abs() < 1e6
axes[1].hexbin(ev['pi_p0'][mask_mu],
               ev['mu_z_decay'][mask_mu] / 1000,
               gridsize=50, cmap='Oranges', mincnt=1)
axes[1].set_xlabel('Initial pi+ momentum (GeV/c)')
axes[1].set_ylabel('mu+ decay z position (m)')
axes[1].set_title('mu+ decay vertex vs initial pi+ momentum')
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.savefig('08_decay_vertices.png', dpi=150)
plt.show()

## 9. Physics process breakdown (what stopped the pion?)

In [ ]:
# Last step of each pi+ track = what process ended it
pi_steps = st[st['particle'] == 'pi+']
last_pi = pi_steps.groupby(['event_id','track_id']).last().reset_index()
proc_counts = last_pi['process'].value_counts()
print('pi+ final processes:')
print(proc_counts)

fig, ax = plt.subplots(figsize=(8, 4))
proc_counts.head(10).plot(kind='barh', ax=ax, color='steelblue', alpha=0.8)
ax.set_xlabel('Count')
ax.set_title('pi+ final step process (decay, absorption, inelastic scatter...)')
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('09_pion_processes.png', dpi=150)
plt.show()

# Fraction that decayed vs were absorbed
n_decay  = proc_counts.get('Decay', 0)
n_absorb = proc_counts.get('pi+Inelastic', 0)
n_total  = proc_counts.sum()
print(f'\nDecay fraction     : {100*n_decay/n_total:.1f}%')
print(f'Inelastic fraction : {100*n_absorb/n_total:.1f}%')

## 10. Summary table

In [ ]:
print('='*60)
print('SIMULATION SUMMARY')
print('='*60)
print(f'Total events             : {len(ev):,}')
print(f'Events with mu+ track    : {(ev["mu_track_len"]>0).sum():,}')
print(f'Events with Michel e+    : {(ev["e_track_len"]>0).sum():,}')
print()
print('pi+:')
print(f'  Mean track length      : {ev["pi_track_len"].mean()/10:.1f} cm')
print(f'  Mean Cherenkov photons : {ev["pi_cher_total"].mean():.0f}')
print(f'  Mean decay time        : {ev["pi_time_decay"].mean():.4f} ns')
print()
print('mu+:')
print(f'  Mean track length      : {ev["mu_track_len"].mean()/10:.1f} cm')
print(f'  Mean Cherenkov photons : {ev["mu_cher_total"].mean():.0f}')
print(f'  Mean decay time        : {ev["mu_time_decay"].mean():.2f} ns')
print()
print('Michel e+:')
print(f'  Mean track length      : {ev["e_track_len"].mean()/10:.1f} cm')
print(f'  Mean Cherenkov photons : {ev["e_cher_total"].mean():.0f}')
print(f'  Mean initial KE        : {ev["e_KE0"].mean()*1000:.1f} MeV')
print()
print('Combined:')
print(f'  Mean total Cherenkov   : {ev["total_cher"].mean():.0f} photons/event')